# Estructura del notebook

 # 1. Imports

In [1]:
# Tools
import os 
import time
import shutil
import random
from typing import Tuple
from argparse import Namespace
import matplotlib.pyplot as plt

# Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk import ngrams
from nltk.tokenize import TweetTokenizer
from nltk import FreqDist
import pandas as pd
import numpy as np

#PyTorch
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F

#scikit-learn
from sklearn.metrics import accuracy_score

 # 2. Random Seeds

In [2]:
seed = 1111
random.seed(seed) # python seed
np.random.seed(seed) # numpy seed
torch.manual_seed(seed) #torch seed

#  Problema 1
Con base en la implementación mostrada en la práctica del modelo de Bengio,
construya un modelo de lenguaje neuronal a nivel de palabra, pero preinicializado con
los embeddings proporcionados. Tomé en cuenta secuencias de tamaño 4 para el modelo,
es decir hasta 3 palabras en el contexto. Después de haber entrenado el modelo, recupere
las 10 palabras más similares a tres palabras de su gusto dadas. Ponga al modelo a
generar texto a partir de tres secuencias de inicio de su gusto. Escriba 5 ejemplos de
oraciones y mídales el likelihood. Proponga un ejemplo para ver estructuras sintácticas
(permutaciones de palabras de alguna oración) buenas usando el likelihood a partir de
una oración que usted proponga. Calcule la perplejidad del modelo sobre los datos val.
Compárelo con la perplejidad del modelo de lenguaje sin embeddings preentrenados y el
probabilista de la tarea anterior (el visto en clase). DISCUTA BREVEMENTE.

# 3. Data Pipeline

### 3. a) Corpus

In [3]:
X_train = pd.read_csv('mex20_train.txt',sep='\r\n',engine='python',header=None).loc[:,0].values
print("TRAIN:",X_train)
X_val = pd.read_csv('mex20_val.txt',sep='\r\n',engine='python',header=None).loc[:,0].values
print("\n\n\nVAL:",X_val)

TRAIN: <ArrowStringArray>
['@USUARIO @USUARIO @USUARIO Q se puede esperar del maricon de closet de la Yañez aun recuerdo esa ves q lo vi en zona rosa viendo quien lo levantada',
                                                       '@USUARIO La piel nueva siempre arde un poquito los primeros días... y más con este puto clima',
                                                                                                           'Ustedes no se enamoran de mí… por tontas.',
                                                                                                      'Me las va a pagar esa puta gorda roba tuits...',
                                                                       '@USUARIO LA GENTE ES TONTA PORQUE NO SE DAN CUENTA QUE TÚ HACES A BATMAN AZUL',
        'Estoy muy encabronada con las pseudo feministas por tontas e iletradas, a veces me avergüenza ser mujer; preferiría tener un falo. #NiUnaMas',
                                      'Anden putos, recuerdan 

### 3. b)NgramData

In [4]:
# --------    NAMESPACE -----------
args = Namespace()

# N 
args.N = 4

class NgramData():
    def __init__(self,N:int, vocab_max : int=5000, tokenizer = None,embeddings_model=None):
        self.tokenizer = tokenizer if tokenizer else self.default_tokenizer 
        self.punct = set(['.',',',';',':','-','^','*','¡','!','?','¿','\'','<unk>','@usuario'])
        self.N = N
        self.vocab_max = vocab_max
        self.embeddings_model = embeddings_model
        self.UNK = '<unk>'
        self.SOS = '<s>'
        self.EOS = '</s>'
        
    def get_vocab_size(self)->int:
        return len(self.vocab)
    def default_tokenizer(self,doc:str)-> list:
        return doc.split(" ")
    
    def get_vocab(self, corpus : list) -> set:
        
        freq_dist = FreqDist([w.lower() for sentence in corpus \
                              for w in self.tokenizer(sentence) \
                              if not self.remove_word(w)]) 
        sorted_words = self.sortFreqDict(freq_dist)[:self.vocab_max-3]
        return set(sorted_words)
        
    def remove_word(self, word: str) -> bool:
        word = word.lower()
        is_punct = word in self.punct 
        is_digit = word.isnumeric()
        return is_punct or is_digit

    def sortFreqDict(self, freq_dict)-> list:
        freq_dict = dict(freq_dict)
        return sorted(freq_dict , key=freq_dict.get, reverse = True)
    
    def fit(self, corpus : list) -> None:
            self.vocab = self.get_vocab(corpus)
            self.vocab.add(self.UNK)
            self.vocab.add(self.SOS)
            self.vocab.add(self.EOS)

            self.w2id = {}
            self.id2w = {}

            if self.embeddings_model is not None:
                 self.embedding_matrix = np.empty([len(self.vocab),self.embeddings_model.vector_size])
            
            id = 0
            for doc in corpus:
                for word in self.tokenizer(doc):
                    word_ = word.lower()
                    if word_ in self.vocab and not word_ in self.w2id:
                        self.w2id[word_] = id
                        self.id2w[id] = word_
                    
                        if self.embeddings_model is not None:
                            if word in self.embeddings_model:
                                self.embedding_matrix[id] = self.embeddings_model[word_]
                            else:
                                self.embedding_matrix[id] = np.random.rand(self.embeddings_model.vector_size)
                        id += 1

        
            # Always add special tokens
            self.w2id.update({
                self.UNK: id,
                self.SOS: id+1,
                self.EOS: id+2
            })
            self.id2w.update({
                id: self.UNK,
                id+1: self.SOS,
                id+2: self.EOS
    
            })

    def transform(self, corpus:list) -> Tuple[np.ndarray,np.ndarray]:
        X_ngrams = []
        y = []

        for doc in corpus:
            doc_ngram = self.get_ngram_doc(doc)
            for words_window in doc_ngram:
                words_window_ids = [self.w2id[w] for w in words_window]
                X_ngrams.append(list(words_window_ids[:-1]))
                y.append(words_window_ids[-1])
                
        return np.array(X_ngrams),np.array(y)

    def get_ngram_doc(self,doc: str) -> list:
        doc_tokens = self.tokenizer(doc)
        doc_tokens = self.replace_unk(doc_tokens)
        doc_tokens = [w.lower() for w in doc_tokens]
        doc_tokens = [self.SOS]*(self.N-1) + doc_tokens + [self.EOS]
        return list(ngrams(doc_tokens,self.N))

    def replace_unk(self, doc_tokens:list)-> list:
        for i, token in enumerate(doc_tokens):
            if token.lower() not in self.vocab:
                doc_tokens[i] = self.UNK
        return doc_tokens
            

###  3. c) Embeddings

In [5]:
# Primer ngram_data.fit(X_train), para el vocabulario
tk = TweetTokenizer()
ngram_data = NgramData(args.N, 5000, tk.tokenize)
ngram_data.fit(X_train)
vocab_words = ngram_data.vocab

pretrained_words = {}
with open("word2vec_col.txt", "r", encoding="utf-8") as f:
    primera_linea = f.readline().strip().split()
    emb_dim = int(primera_linea[1])
    print(f"Dimensión de embeddings: {emb_dim}")

    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        if word not in vocab_words:  
            continue
        try:
            vec = np.array(parts[1:], dtype=np.float32)
            if len(vec) == emb_dim:
                pretrained_words[word] = vec
        except ValueError:
            continue
            
class EmbeddingsWrapper:
    def __init__(self, word_vectors, dim):
        self._vecs = word_vectors
        self.vector_size = dim
    def __contains__(self, word):
        return word in self._vecs
    def __getitem__(self, word):
        return self._vecs[word]

embeddings_model = EmbeddingsWrapper(pretrained_words, emb_dim)
ngram_data = NgramData(args.N, 5000, tk.tokenize, embeddings_model=EmbeddingsWrapper(pretrained_words, emb_dim))
ngram_data.fit(X_train)
print(f"embedding_matrix shape: {ngram_data.embedding_matrix.shape}")  # debe ser (5000, 100)

Dimensión de embeddings: 100
embedding_matrix shape: (5000, 100)


### 3. d) DataLoader (batches) 

In [6]:
# Segundo ngram_data.fit(X_train), ahora con los embeddings cargados
tk = TweetTokenizer()
ngram_data = NgramData(args.N, 5000, tk.tokenize, embeddings_model=embeddings_model)
ngram_data.fit(X_train)

X_ngram_train, y_ngram_train = ngram_data.transform(X_train)
X_ngram_val, y_ngram_val = ngram_data.transform(X_val)
X_ngram_train,y_ngram_train


#Set batch size in args
args.batch_size = 64

#Num workers
args.num_workers = 2

#Train
train_dataset = TensorDataset(torch.tensor(X_ngram_train, dtype = torch.int64), torch.tensor(y_ngram_train, dtype = torch.int64))

train_loader =DataLoader(train_dataset, batch_size=args.batch_size, num_workers = args.num_workers, shuffle=True)

#Val
val_dataset = TensorDataset(torch.tensor(X_ngram_val, dtype = torch.int64), torch.tensor(y_ngram_val, dtype = torch.int64))

val_loader =DataLoader(val_dataset, batch_size=args.batch_size, num_workers = args.num_workers, shuffle=False)

# 4. Bengio's Model 

In [7]:
#Vocab size
args.vocab_size = ngram_data.get_vocab_size()

# Dimension of words embeddings
args.d = 50

# Dimension for hidden layer
args.d_h = 160

In [8]:
class BengioLM(nn.Module):

    def __init__(self, args):
        super(BengioLM, self).__init__()

        self.window_size = args.N-1
        self.embedding_size = args.d

        # Inicialización de embeddings word2vec_col
        self.emb = nn.Embedding(args.vocab_size, args.d)        
        self.fc1 = nn.Linear(args.d*(args.N-1), args.d_h)  # y = Wx + b
        self.drop1 = nn.Dropout(p=args.dropout)
        self.fc2 = nn.Linear(args.d_h, args.vocab_size, bias=False) 

    def forward(self, x):

        x = self.emb(x)
        x = x.view(-1, self.window_size * self.embedding_size)
        
        h = torch.tanh(self.fc1(x)) 
        h = self.drop1(h)
        return self.fc2(h)        

# 5. Train functions 

In [9]:
def get_preds(raw_logits):
    probs = F.softmax(raw_logits.detach(),dim=1)
    y_pred = torch.argmax(probs, dim=1).cpu().numpy()
    return y_pred
    
def model_eval(data,model,gpu=False):
    with torch.no_grad():
        preds, tgts = [],[]
        for window_words,labels in data:
            if gpu:
                window_words = window_words.cuda()
    
            outputs = model(window_words)
    
            #Get predicitons
            y_pred = get_preds(outputs)
    
            tgt = labels.numpy()
            tgts.append(tgt)
            preds.append(y_pred)
        tgts = [e for l in   tgts for e in l]
        preds = [e for l in preds for e in l]
        return accuracy_score(tgts, preds)
def save_checkpoint(state, is_best, checkpoint_path, filename="checkpoint.pt"): 
    me = os.path.join(checkpoint_path, filename)
    torch.save(state, me)
    if is_best:
        shutil.copyfile(me, os.path.join(checkpoint_path, "model_best.pt")) 

# 6. Hyperparameters

In [38]:
# Model hyperparameters
args.vocab_size = ngram_data.get_vocab_size()
args.d = 100 # Dimension of word embedding
args.d_h = 5000 # Dimension for hidden layer
args.dropout = 0.3

#Trainning hyperparameters
args.lr = 1e-3
args.num_epochs = 100
args.patience = 20

#Scheduler hyperparameters
args.lr_patience = 10
args.lr_factor = 0.5

#Saving directory
args.savedir = 'model'
os.makedirs(args.savedir, exist_ok = True)

#Create model
model = BengioLM(args)

#Embeddings 
emb_tensor = torch.FloatTensor(ngram_data.embedding_matrix)
model.emb.weight = nn.Parameter(emb_tensor)

#Loss, Optimizer and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer=optimizer,  
                mode="min",
                patience=args.lr_patience,
                factor=args.lr_factor
            )


# 7. Trainning

In [39]:
def training_loop(save_name, model, optimizer, scheduler, criterion,train_loader, val_loader, args):

    # Verificación rápida de datos y modelo
    for window_words, labels in train_loader:
        print("window_words shape:", window_words.shape)
        print("labels shape:", labels.shape)
        print("labels min/max:", labels.min().item(), labels.max().item())
        print("ejemplo window_words:", window_words[0])
        print("ejemplo label:", labels[0])
        break
    print("vocab_size:", args.vocab_size)
    print("N:", args.N)

    start_time = time.time()
    best_metric = 0
    metric_history = []
    train_metric_history = []
    n_no_improve = 0

    for epoch in range(args.num_epochs):
        epoch_start_time = time.time()
        loss_epoch = []
        training_metric = []
        model.train()

        for window_words, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(window_words)
            loss = criterion(outputs, labels)
            loss_epoch.append(loss.item())

            y_pred = get_preds(outputs)
            tgt = labels.cpu().numpy()
            training_metric.append(accuracy_score(tgt, y_pred))

            loss.backward()
            optimizer.step()

        mean_epoch_metric = np.mean(training_metric)
        train_metric_history.append(mean_epoch_metric)

        model.eval()
        tuning_metric = model_eval(val_loader, model, gpu=False)
        metric_history.append(tuning_metric)

        scheduler.step(tuning_metric)

        is_improvement = tuning_metric > best_metric
        if is_improvement:
            best_metric = tuning_metric
            n_no_improve = 0
        else:
            n_no_improve += 1

        save_checkpoint(
            {
                "epoch": epoch + 1,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_metric": best_metric,
            },
            is_improvement,
            args.savedir,
            filename=f"{save_name}_checkpoint.pt"  
        )

        print(f"Epoch [{epoch+1}/{args.num_epochs}] | Loss {np.mean(loss_epoch):.4f} | Train acc: {mean_epoch_metric:.4f} | Val acc: {tuning_metric:.4f} | Epoch time: {time.time()-epoch_start_time:.4f}")

        if n_no_improve >= args.patience:
            print("No improvement, Breaking out of loop.")
            break

    print("... %s seconds " % (time.time() - start_time))
    return metric_history, train_metric_history


In [40]:
history_sgd, train_history_sgd = training_loop("bengio_sgd", model, optimizer, scheduler,criterion, train_loader, val_loader, args)

window_words shape: torch.Size([64, 3])
labels shape: torch.Size([64])
labels min/max: 0 4999
ejemplo window_words: tensor([4997,  772, 4979])
ejemplo label: tensor(777)
vocab_size: 5000
N: 4
Epoch [1/100] | Loss 7.0478 | Train acc: 0.1624 | Val acc: 0.2045 | Epoch time: 96.6374
Epoch [2/100] | Loss 6.2784 | Train acc: 0.1792 | Val acc: 0.2130 | Epoch time: 97.8874
Epoch [3/100] | Loss 5.9892 | Train acc: 0.1861 | Val acc: 0.2180 | Epoch time: 96.7152
Epoch [4/100] | Loss 5.7954 | Train acc: 0.1897 | Val acc: 0.2224 | Epoch time: 96.5251
Epoch [5/100] | Loss 5.6463 | Train acc: 0.1928 | Val acc: 0.2265 | Epoch time: 96.5370
Epoch [6/100] | Loss 5.5213 | Train acc: 0.1954 | Val acc: 0.2292 | Epoch time: 100.6378
Epoch [7/100] | Loss 5.4161 | Train acc: 0.1973 | Val acc: 0.2298 | Epoch time: 100.6538
Epoch [8/100] | Loss 5.3288 | Train acc: 0.1967 | Val acc: 0.2270 | Epoch time: 99.8546
Epoch [9/100] | Loss 5.2518 | Train acc: 0.1978 | Val acc: 0.2307 | Epoch time: 99.5891
Epoch [10/100]

# 8. Evaluation

In [47]:
#  Cargar mejor modelo 
best_model = BengioLM(args)
best_model.load_state_dict(torch.load('model/model_best.pt')['state_dict'])
best_model.train(False)
best_model.eval()

BengioLM(
  (emb): Embedding(5000, 100)
  (fc1): Linear(in_features=300, out_features=5000, bias=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=5000, out_features=5000, bias=False)
)

### 8. 1) Sentence Generation

In [48]:
def parse_text(text, tokenizer):
    all_tokens = [w.lower() if w in ngram_data.w2id else '<unk>' for w in tokenizer.tokenize(text)]
    token_ids = [ngram_data.w2id[word.lower()] for word in all_tokens]
    return all_tokens, token_ids
    
def sample_next_word(logits, temperature=1.0):
    logits = np.asarray(logits).astype('float64')
    preds = logits / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds)
    return np.argmax(probas)
    
def predict_next_token(model, token_ids, temperature = 1.0):
    word_ids_tensor = torch.LongTensor(token_ids).unsqueeze(0)
    y_raw_pred = model(word_ids_tensor).squeeze(0).detach().numpy()
    
    # el segundo parametroe es un parámetro que regula que tan determinista-aleatorio lo queremos ()
    y_pred = sample_next_word(y_raw_pred, temperature = temperature) 

    return y_pred

def generate_sentence(model, initial_text, tokenizer, temperature = 1.0):
    all_tokens, window_word_ids = parse_text(initial_text, tokenizer)

    for i in range(100):
        y_pred = predict_next_token(best_model, window_word_ids, temperature)
        next_word = ngram_data.id2w[y_pred]
        all_tokens.append(next_word)

        if next_word == '</s>':
            break
        else:
            window_word_ids.pop(0)
            window_word_ids.append(y_pred)
    return " ".join(all_tokens)

In [49]:
initial_tokens_list = [
    "más vale que",
    "y por que",
    "yo creo que",
    "ojalá un día",
    "qué onda con",
    "me vale la",
    "ya no puedo",
    "eso no se",
    "la verdad es",
    "por qué siempre"
]

print("."*30)
print("Sentence Generation")
print("."*30)
for initial_tokens in initial_tokens_list:
    print(f"\n-- Inicio: '{initial_tokens}' --")
    print(generate_sentence(best_model, initial_tokens, tk))

..............................
Sentence Generation
..............................

-- Inicio: 'más vale que' --
más vale que que se <unk> " de lluvia <unk> qué caso de que tonta #gorda han dicho eso mames dirán <unk> </s>

-- Inicio: 'y por que' --
y por que nada <unk> a el hijo a los celebrando y los roba que la <unk> <unk> 😍 se le amo <unk> apesta gorda sabe pero vida <unk> </s>

-- Inicio: 'yo creo que' --
yo creo que 🙄 que todas calidad de tweets <unk> en sólo mundo así en el q <unk> soy ya sí he <unk> acciones <unk> <unk> y cual <unk> amor no pueden creer te xd bolsa <unk> </s>

-- Inicio: 'ojalá un día' --
ojalá un día jsjs sólo bonita valga verga no me <unk> <unk> la misma pesos y <unk> 👏 twitter <unk> se <unk> cobijas y me se le verdes perros — <unk> </s>

-- Inicio: 'qué onda con' --
qué onda con los ando pues viendo vayanse maria </s>

-- Inicio: 'me vale la' --
me vale la verga jajaja estoy que fea vamos a ir a salen y la verga me </s>

-- Inicio: 'ya no puedo' --
ya no pued

### 8.  2) Syntactic Permutations

In [50]:
from itertools import permutations
from random import shuffle

def log_likelihood(model, text, ngram_model):
    # Generate n-gram windows from input textand the respective label y
    X,y = ngram_data.transform([text])
    X,y = X[2:],y[2:]
    X = torch.LongTensor(X).unsqueeze(0)

    logits = model(X).detach()
    probs = F.softmax(logits, dim=1).numpy()

    return np.sum([np.log(probs[i][w]) for i,w in enumerate(y)])

def show_syntactic_permutations(sentence, model, ngram_data, top_n=5):
    word_list = sentence.split(' ')
    perms = [' '.join(perm) for perm in permutations(word_list)]
    log_likelihoods = sorted(
        [(log_likelihood(model, text, ngram_data), text) for text in perms],
        reverse=True
    )
    print('\n', '.'*25, "Most probable", '.'*25)
    for p, t in log_likelihoods[:top_n]:
        print("L(*):", p, " Oracion: ", t)
    print('\n', '.'*25, "Less probable", '.'*25)
    for p, t in log_likelihoods[-top_n:]:
        print("L(*):", p, " Oracion: ", t)

In [51]:
show_syntactic_permutations("sino gano me voy a la chingada", best_model, ngram_data)


 ......................... Most probable .........................
L(*): -21.823654  Oracion:  gano sino voy a la chingada me
L(*): -22.539186  Oracion:  gano sino me voy a la chingada
L(*): -22.634361  Oracion:  sino gano voy a la chingada me
L(*): -23.216133  Oracion:  voy gano a la chingada sino me
L(*): -23.264515  Oracion:  voy sino gano a la chingada me

 ......................... Less probable .........................
L(*): -50.50262  Oracion:  me la a sino gano chingada voy
L(*): -50.64607  Oracion:  la me sino a chingada gano voy
L(*): -51.044518  Oracion:  me a chingada sino gano voy la
L(*): -51.11533  Oracion:  me a chingada gano voy sino la
L(*): -51.136925  Oracion:  me la a chingada sino gano voy


 ### 8. 3) Perplexity

In [52]:
def perplexity(model, data_loader):
    model.eval()
    total_log_prob = 0
    total_words = 0

    with torch.no_grad():
        for window_words, labels in data_loader:
            logits = model(window_words)
            probs = F.softmax(logits, dim=1)

            # prob de la palabra correcta para cada ejemplo del batch
            correct_probs = probs[torch.arange(len(labels)), labels]
            total_log_prob += torch.log(correct_probs).sum().item()
            total_words += len(labels)

    return np.exp(-total_log_prob / total_words)


In [53]:
ppl = perplexity(best_model, val_loader)
print(f"Perplexidad en val: {ppl:.4f}")

Perplexidad en val: 114.1161


#  Problema  2
Con base en la implementación mostrada en las prácticas del NLM, construya un
modelo de lenguaje neuronal a nivel de carácter. Tomé en cuenta secuencias de tamaño 6
o más para el modelo, es decir hasta 5 caracteres o más en el contexto. Ponga al modelo a
generar texto 3 veces, con un máximo de 300 caracteres. Escriba 5 ejemplos de oraciones
y mídales el likelihood. Escriba un ejemplo de estructura morfológica (permutaciones
con caracteres) similar al de estructura sintáctica del profesor con 5 o más caracteres de
su gusto (e.g., "ando "). Calcule la perplejidad del modelo sobre los datos val. DISCUTA
BREVEMENTE.

# 9. CharData

In [38]:
class CharData():
    def __init__(self, N:int):
        self.N = N
        self.UNK = '<unk>'
        self.SOS = '<s>'
        self.EOS = '</s>'

    def get_vocab_size(self):
        return len(self.w2id)

    def fit(self, corpus:list):
        # vocabulario = todos los caracteres únicos
        all_chars = set(c for doc in corpus for c in doc)
        self.vocab = all_chars
        self.vocab.add(self.UNK)
        self.vocab.add(self.SOS)
        self.vocab.add(self.EOS)

        self.w2id = {c:i for i,c in enumerate(sorted(self.vocab))}
        self.id2w = {i:c for c,i in self.w2id.items()}
        print(f"Vocab size (chars): {self.get_vocab_size()}")

    def transform(self, corpus:list):
        X_ngrams, y = [], []
        for doc in corpus:
            tokens = [self.SOS]*(self.N-1) + list(doc) + [self.EOS]
            tokens = [c if c in self.w2id else self.UNK for c in tokens]
            for ngram in ngrams(tokens, self.N):
                ids = [self.w2id[c] for c in ngram]
                X_ngrams.append(ids[:-1])
                y.append(ids[-1])
        return np.array(X_ngrams), np.array(y)

# 10. Data Pipeline

In [203]:
# 1 CharData fit
args_char = Namespace()
args_char.N = 6  # 5 caracteres de contexto

char_data = CharData(args_char.N)
char_data.fit(X_train)
print(f"Vocab size (chars): {char_data.get_vocab_size()}")

# 2 Transform 
X_ngram_train_char, y_ngram_train_char = char_data.transform(X_train)
X_ngram_val_char,   y_ngram_val_char   = char_data.transform(X_val)
print(f"Train shape: {X_ngram_train_char.shape}")
print(f"Val shape:   {X_ngram_val_char.shape}")

#  3 DataLoader 
args_char.batch_size  = 256  # caracteres son más ligeros, se puedes hacer batches más grande
args_char.num_workers = 2

train_dataset_char = TensorDataset(
    torch.tensor(X_ngram_train_char, dtype=torch.int64),
    torch.tensor(y_ngram_train_char, dtype=torch.int64)
)
val_dataset_char = TensorDataset(
    torch.tensor(X_ngram_val_char, dtype=torch.int64),
    torch.tensor(y_ngram_val_char,  dtype=torch.int64)
)

train_loader_char = DataLoader(
    train_dataset_char, batch_size=args_char.batch_size,
    num_workers=args_char.num_workers, shuffle=True
)
val_loader_char = DataLoader(
    val_dataset_char, batch_size=args_char.batch_size,
    num_workers=args_char.num_workers, shuffle=False
)

print(f"Train batches: {len(train_loader_char)}")
print(f"Val batches:   {len(val_loader_char)}")

Vocab size (chars): 428
Vocab size (chars): 428
Train shape: (468758, 5)
Val shape:   (52518, 5)
Train batches: 1832
Val batches:   206


# 11. Hyperparameters

In [ ]:
args_char.vocab_size = char_data.get_vocab_size()  
args_char.d         = 30       
args_char.d_h       = 800  
args_char.dropout   = 0.3

args_char.lr         = 1e-3
args_char.num_epochs = 100
args_char.patience   = 20
args_char.lr_patience = 10
args_char.lr_factor  = 0.5
args_char.use_gpu    = torch.cuda.is_available()
args_char.savedir    = 'model_char'

# 12. Trainning

In [205]:
os.makedirs(args_char.savedir, exist_ok=True)

# Modelo: Se puede reutilizar esta clase solo con el Namespace de args_char
model_char = BengioLM(args_char)

# Optimizer y Scheduler
criterion_char = nn.CrossEntropyLoss()
optimizer_char = torch.optim.Adam(model_char.parameters(), lr=args_char.lr)

scheduler_char = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer_char, mode="min",
    patience=args_char.lr_patience, factor=args_char.lr_factor
)

#  Trainning 
history_char, train_history_char = training_loop(
    "bengio_char", model_char,
    optimizer_char, scheduler_char,
    criterion_char, train_loader_char, val_loader_char, args_char
)

window_words shape: torch.Size([256, 5])
labels shape: torch.Size([256])
labels min/max: 0 341
ejemplo window_words: tensor([117,   0,  48, 114,  89])
ejemplo label: tensor(74)
vocab_size: 428
N: 6
Train acc : 0.36850483202659784
Epoch [1/100], Loss 2.2866 - Val accuracy: 0.4275 - Epoch time: 13.8922
Train acc : 0.42048982421099645
Epoch [2/100], Loss 2.0530 - Val accuracy: 0.4545 - Epoch time: 13.5159
Train acc : 0.4389249128498412
Epoch [3/100], Loss 1.9797 - Val accuracy: 0.4646 - Epoch time: 13.5029
Train acc : 0.45150023881004364
Epoch [4/100], Loss 1.9321 - Val accuracy: 0.4745 - Epoch time: 13.8041
Train acc : 0.4595817877878126
Epoch [5/100], Loss 1.8991 - Val accuracy: 0.4835 - Epoch time: 13.7252
Train acc : 0.4659100601056967
Epoch [6/100], Loss 1.8738 - Val accuracy: 0.4883 - Epoch time: 13.6180
Train acc : 0.4709595744218936
Epoch [7/100], Loss 1.8529 - Val accuracy: 0.4906 - Epoch time: 13.6851
Train acc : 0.47571406529128624
Epoch [8/100], Loss 1.8372 - Val accuracy: 0.4

# 13. Evaluation

In [206]:
# Cargar  modelo char 
best_model_char = BengioLM(args_char)
best_model_char.load_state_dict(
    torch.load('model_char/model_best.pt')['state_dict']
)
best_model_char.eval()

BengioLM(
  (emb): Embedding(428, 30)
  (fc1): Linear(in_features=150, out_features=800, bias=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=800, out_features=428, bias=False)
)

### 13. a) Char Generation

In [207]:
def parse_text_char(text, char_data):
    all_tokens = [c if c in char_data.w2id else '<unk>' for c in text]
    token_ids = [char_data.w2id[c] for c in all_tokens]
    return all_tokens, token_ids

def generate_text_char(model, initial_text, char_data, temperature=1.0, max_chars=300):
    all_tokens, window_ids = parse_text_char(initial_text, char_data)
    
    # solo tomamos los últimos N-1 como ventana
    window_ids = window_ids[-(char_data.N-1):]
    
    for i in range(max_chars):
        y_pred = predict_next_token(model, window_ids, temperature)
        next_char = char_data.id2w[y_pred]
        all_tokens.append(next_char)
        
        if next_char == '</s>':
            break
        else:
            window_ids.pop(0)
            window_ids.append(y_pred)
    return "".join(all_tokens)

In [262]:
oraciones = [
    "   des" ,
    "   pre" ,
    "    re", 
    "    in",   
    "   con" ,
    "   entr" 
]
for oracion in oraciones:
    print("\n")
    print("_"*30)
    print(f"ORACIÓN : {oracion}")
    for i in range(3):
        print(f"\n── Generación {i+1} ──")
        print(generate_text_char(best_model_char, oracion, char_data))



______________________________
ORACIÓN :    des

── Generación 1 ──
   desmos no sorra</s>

── Generación 2 ──
   desdo pullos #Hombro esuia semana hacen zotisino, albaga alha ificiente oso</s>

── Generación 3 ──
   deseando trincciFa.</s>


______________________________
ORACIÓN :    pre

── Generación 1 ──
   preazo de lo sabe de Grucho de verga :y</s>

── Generación 2 ──
   presumlotes pay  otro me vale verga.</s>

── Generación 3 ──
   preser pique detele a regrenza y verdad sccial onicas putas tontas, hace dejombres del no se puede unarte deMa de la verga!</s>


______________________________
ORACIÓN :     re

── Generación 1 ──
    reanos no se el"nó si fuoriotorialibes de hace un chingas amos, el artejo te di #buros una vierno no indagramos corta</s>

── Generación 2 ──
    reguada coasilandumentario estoy pessi en!</s>

── Generación 3 ──
    reot comalta el para nos te cuanto una de le tenía V" playa</s>


______________________________
ORACIÓN :     in

── Generación 1 ──


### 13. b) Sentence Examples

In [243]:
oraciones = [
    "vete a la verga",
    "no manches, me cae mal",
    "yo creo que sí",
    "haber caminado por aquí",
    "ya no puedo más",
    "cabra te tu no ",
    "me la pelan"
]
for oracion in oraciones:
    ll = log_likelihood_char(best_model_char, oracion, char_data)
    print(f"L(*): {ll:.4f}  |  '{oracion}'")

L(*): -20.1589  |  'vete a la verga'
L(*): -47.1973  |  'no manches, me cae mal'
L(*): -26.8594  |  'yo creo que sí'
L(*): -43.7849  |  'haber caminado por aquí'
L(*): -23.3997  |  'ya no puedo más'
L(*): -42.6526  |  'cabra te tu no '
L(*): -17.8069  |  'me la pelan'


### 13. c) Morphological Premutations

In [221]:
def log_likelihood_char(model, text, char_data):
    X, y = char_data.transform([text])
    X = torch.LongTensor(X).unsqueeze(0)
    logits = model(X).detach()
    probs = F.softmax(logits, dim=1).numpy()
    return np.sum([np.log(probs[i][w]) for i, w in enumerate(y)])

def show_morphological_permutations(chars, model, char_data, top_n=5):
    # permutaciones de caracteres, no palabras
    char_list = list(chars)
    perms = [''.join(perm) for perm in permutations(char_list)]
    perms = list(set(perms))  # eliminar duplicados
    log_likelihoods = sorted(
        [(log_likelihood_char(model, text, char_data), text) for text in perms],
        reverse=True
    )
    print('\n', '.'*25, "Most probable", '.'*25)
    for p, t in log_likelihoods[:top_n]:
        print("L(*):", p, " Chars: ", t)
    print('\n', '.'*25, "Less probable", '.'*25)
    for p, t in log_likelihoods[-top_n:]:
        print("L(*):", p, " Chars: ", t)

In [225]:
show_morphological_permutations("ando", model_char, char_data)


 ......................... Most probable .........................
L(*): -14.121659  Chars:  ando
L(*): -15.170481  Chars:  nado
L(*): -17.911594  Chars:  onda
L(*): -18.189968  Chars:  adon
L(*): -20.111225  Chars:  dona

 ......................... Less probable .........................
L(*): -29.106777  Chars:  naod
L(*): -29.207401  Chars:  noad
L(*): -29.653522  Chars:  aodn
L(*): -30.82457  Chars:  dnao
L(*): -31.879782  Chars:  dnoa


### 13. d) Perplexity

In [231]:
ppl = perplexity(best_model_char, val_loader_char)
print(f"Perplexidad en val: {ppl:.4f}")

Perplexidad en val: 5.3823


In [214]:
print("log-likelihood: ", log_likelihood(best_model, "Ahora si, creo que voy a decir que no",ngram_data))

log-likelihood:  -43.64923


In [215]:
print("log-likelihood: ", log_likelihood(best_model, "Estamos en la clase de procesamiento del lenguaje natural",ngram_data))

log-likelihood:  -34.885494
